# ARS Notebook 5 - Router Diagnostics and Ablations

Read saved four-arm results only. This notebook audits the router's route, active-layer-count, and alpha distributions alongside paired accuracy. It does not retrain the router or run model inference.


## Input contract

Attach any completed result bundles from Notebooks 2-4: `ars_gsm8k_results.zip`, `ars_math_ood_results.zip`, and/or `ars_mcq_results.zip`. Attach multiple bundles together when available. Notebook 1's model bundle is not required for this read-only analysis.

In [ ]:
import json, os, subprocess, sys, zipfile, shutil
from pathlib import Path
os.environ["TOKENIZERS_PARALLELISM"] = "false"
REPO_URL = "https://github.com/Abdullah182155/adaptive-residual-steering.git"
REPO_DIR = Path("/kaggle/working/adaptive-residual-steering")
WORK_ROOT = Path("/kaggle/working")
INPUT_ROOT = Path("/kaggle/input")
if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "development"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "development"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", "development"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", "development", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
bundle_names = ("ars_gsm8k_results.zip", "ars_math_ood_results.zip", "ars_mcq_results.zip")
bootstrap_archives = [p for name in bundle_names for p in INPUT_ROOT.rglob(name)]
code_bootstrapped = False
for bootstrap_archive in bootstrap_archives:
    with zipfile.ZipFile(bootstrap_archive) as archive:
        if "code/scripts/research_notebook_utils.py" not in archive.namelist() or "code/scripts/research_notebook_analysis.py" not in archive.namelist():
            continue
        for name in ("research_notebook_utils.py", "research_notebook_analysis.py"):
            member = f"code/scripts/{name}"
            (REPO_DIR / "scripts" / name).write_bytes(archive.read(member))
        code_bootstrapped = True
        break
if not code_bootstrapped:
    raise FileNotFoundError("Attach a result bundle created by the new Notebook 2, 3, or 4; it carries the matching analysis code.")
from scripts.research_notebook_analysis import merge_available_results, build_router_diagnostics, create_paper_assets
RESULTS_ROOT = WORK_ROOT / "ars_all_saved_results"
INPUT_REPORT = merge_available_results([INPUT_ROOT], RESULTS_ROOT)
print(INPUT_REPORT)

## Diagnostics

The output reports unique routes, route entropy, top-route share, k counts, layer counts, alpha counts, arm accuracy, and paired router comparisons. Diversity alone is not evidence of improvement; interpret it with the paired score and interval.

In [ ]:
DIAGNOSTIC_DIR = WORK_ROOT / "ars_router_diagnostics"
diagnostics = build_router_diagnostics([RESULTS_ROOT], DIAGNOSTIC_DIR)
print({"accuracy_conditions": len(diagnostics["accuracy_rows"]) // 4,
       "paired_comparisons": len(diagnostics["paired_rows"]),
       "router_conditions": len(diagnostics["route_rows"]),
       "json": str(DIAGNOSTIC_DIR / "router_diagnostics.json"),
       "tables": [str(p) for p in DIAGNOSTIC_DIR.glob("*.csv")]})

## Ablation boundary

The four arms compare baseline, fixed-route low-rank residual, static bank action, and router. Training ablations such as route-cost changes require separate training runs and validation-selected checkpoints; this notebook does not tune on these test results.